In [ ]:
## Pothan Tang, 8/15/25
## Access the functions in Potential Class to calculate ion positions, normal modes, eigenfrequencies 
import constants as c
import math
from Potential import Potential
import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from scipy.optimize import minimize as scipy_minimize
from autograd import hessian as ag_hessian
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error

## region of interest
um = c.um
res = 1*um; # resolution of potential
x_max = 100*um;
y_max = 100*um;
z_max = 200*um;
x = np.linspace(-1*x_max,x_max, int(2*x_max/res+1), endpoint=True); 
y = np.linspace(-1*y_max,y_max, int(2*y_max/res+1), endpoint=True); 
z = np.linspace(0,z_max, int(z_max/res+1), endpoint=True); 

# Parameters for the 3-ion chain
num_ions = 3
mass_amu = 171 # Mass of Ytterbium-171 ion
AMU = 1.66053907e-27  # converts AMU to kg
potential_order = 2 # For example, assuming a quadratic axial potential. Adjust if fitting higher orders.
K = 8.9875e9          # Coulomb's constant
CHARGE = 1.602176634e-19 # Elementary charge
m = c.m

# 1. INPUT: fsec value needed for the Potential class constructor
# trap frequencies (Hz)
freq_x = 719430.7131391969; #718896.9797697717
freq_y = 3031200.0099101723; #3031238.031559673
freq_z = 3002153.5607483205; #3027443.948446284
fsec_for_init_mhz = freq_y/1e6; #3.03; #3.052; # 2.575641; #3.57587

# 2. Initialize Potential object with the derived fsec
ion_trap = Potential(n_ion=num_ions, mass=mass_amu, order=potential_order, fsec=fsec_for_init_mhz)

# 3. INPUT: Trap coefficients C, from V = 0.5 * C * x^2 (in SI units)
fitted_C_x_SI = m*(2*math.pi*freq_x)**2;#target ~#5.86e-12 or 0.719 MHz
fitted_C_y_SI = m*(2*math.pi*freq_y)**2;
fitted_C_z_SI = m*(2*math.pi*freq_z)**2; 
print(f"Trap coefficients x,y,z: {fitted_C_x_SI}, {fitted_C_y_SI}, {fitted_C_z_SI}")

# 4. Set the characteristic length scale. 
ion_trap.set_length_scale()

# 5. Convert fitted SI coefficients to unitless coefficients.
# C_unitless = C_SI * (length_scale^3) / (K * CHARGE^2)
conversion_factor = (ion_trap.length_scale**3) / (K * CHARGE**2)

C_x_unitless = fitted_C_x_SI * conversion_factor
C_y_unitless = fitted_C_y_SI * conversion_factor
C_z_unitless = fitted_C_z_SI * conversion_factor

# 6. Create the full coefficient matrix to pass to set_coeffs.
# Quadratic terms are at index 1 for each dimension (0=x, 1=y, 2=z).
new_potential_coeff_matrix = np.zeros((ion_trap.d, ion_trap.order))

# Set the axial (X) coefficient manually
new_potential_coeff_matrix[0, 1] = C_x_unitless

# Call set_coeffs
ion_trap.set_coeffs(new_potential_coeff_matrix)

# 7. Calculate equilibrium positions for this potential
ion_trap.equilibrium_pos()

# 8. Calculate the Hessian, eigenfrequencies, and motional modes
ion_trap.calc_hessian()

# 9. Retrieve and print the final results

# Equilibrium Ion Positions (in meters, scaled by length_scale)
eq_positions = ion_trap.get_pos()
print("\nEquilibrium Ion Positions (meters):")
for i in range(num_ions):
    print(f"  Ion {i+1}: X={eq_positions[0, i]:.10e} m, Y={eq_positions[1, i]:.10e} m, Z={eq_positions[2, i]:.10e} m")

# Eigenfrequencies and Motional Modes
_, eigenfrequencies, eigenvectors = ion_trap.get_mode()

print("\nEigenfrequencies (Hz):")
# Sort frequencies for clearer output.
sorted_indices = np.argsort(np.real(eigenfrequencies))
sorted_frequencies = np.real(eigenfrequencies[sorted_indices])

for i, freq in enumerate(sorted_frequencies):
    print(f"  Mode {i+1}: {freq:.2f} Hz")

print("\nMotional Modes (Eigenvectors - normalized):")
print("Each column is an eigenvector representing a mode.")
print("The rows correspond to the displacements (x1, x2, x3, y1, y2, y3, z1, z2, z3).")
sorted_eigenvectors = eigenvectors[:, sorted_indices]
np.set_printoptions(precision=4, suppress=True, linewidth=150)
print(sorted_eigenvectors)

# Example of getting transverse modes specifically (using the provided method)
print("\n--- Transverse Normal Modes (Y-direction) ---")
transverse_freqs, transverse_modes = ion_trap.transverse_normal_modes()
print("\nTransverse Eigenfrequencies (Hz):")
for i, freq in enumerate(transverse_freqs):
    print(f"  Transverse Mode {i+1}: {freq:.2f} Hz")

print("\nTransverse Motional Modes (Eigenvectors for Y-direction modes):")
print(transverse_modes)

Trap coefficients x,y,z: 5.802069064063192e-12, 1.0299926671555869e-10, 1.0103474528336778e-10
Your RF is 0.00017958268027376775

Equilibrium Ion Positions (meters):
  Ion 1: X=-3.7066438742e-06 m, Y=0.0000000000e+00 m, Z=0.0000000000e+00 m
  Ion 2: X=-4.9903032463e-14 m, Y=0.0000000000e+00 m, Z=0.0000000000e+00 m
  Ion 3: X=3.7066438511e-06 m, Y=0.0000000000e+00 m, Z=0.0000000000e+00 m

Eigenfrequencies (Hz):
  Mode 1: 719430.71 Hz
  Mode 2: 1246090.55 Hz
  Mode 3: 1732618.60 Hz
  Mode 4: 2818861.50 Hz
  Mode 5: 2818861.50 Hz
  Mode 6: 2944587.06 Hz
  Mode 7: 2944587.06 Hz
  Mode 8: 3031200.01 Hz
  Mode 9: 3031200.01 Hz

Motional Modes (Eigenvectors - normalized):
Each column is an eigenvector representing a mode.
The rows correspond to the displacements (x1, x2, x3, y1, y2, y3, z1, z2, z3).
[[ 0.5774  0.7071 -0.4082  0.      0.      0.      0.      0.      0.    ]
 [ 0.5774  0.      0.8165  0.      0.      0.      0.      0.      0.    ]
 [ 0.5774 -0.7071 -0.4082  0.      0.      0. 

#### Simulation Results
Traverse eigenfrequencies for y: 
*  Transverse Mode 1: 3031200.01 Hz
*  Transverse Mode 2: 2944587.06 Hz
*  Transverse Mode 3: 2818861.50 Hz

Traverse eigenfrequencies for z: 
*  Transverse Mode 1: 3002153.56 Hz
*  Transverse Mode 2: 2914677.59 Hz
*  Transverse Mode 3: 2787603.39 Hz

Target radial eigenfrequencies:
*  Transverse Mode 1: 3030000 Hz
*  Transverse Mode 2: 2942000 Hz
*  Transverse Mode 3: 2817000 Hz